# 4.20 Subida Masiva de Predicciones a Kaggle (Google Colab)

Este notebook está diseñado para ejecutarse desde **Google Colab**.

### Objetivo:
1. Conectar con Google Drive y configurar las credenciales de la API de Kaggle (`kaggle.json`).
2. Recorrer los archivos `.csv` del directorio `procesamiento_local` correspondientes al experimento especificado (por ejemplo `exp420_00`).
3. Subir cada predicción a la competencia de Kaggle mediante la CLI (`kaggle competitions submit`).
4. Una vez subido con éxito, renombrar o marcar el archivo agregándole un prefijo (por defecto `subido_` o `zz_`) para evitar reenvíos duplicados en futuras ejecuciones.

#### 1. Seteo del ambiente en Google Colab

Esta primera parte se debe correr con el runtime en **Python 3**:
<br>Ir al menú: `Runtime` -> `Change runtime type` -> `Python 3`

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/.drive')

Configurar carpetas en Google Drive e instalar credenciales de `kaggle.json` en la máquina virtual de Colab.

In [ ]:
%%shell

# Crear carpetas de sincronización
mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf" /content/buckets/b1

# Instalar credenciales de Kaggle
mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json ~/.kaggle/
chmod 600 ~/.kaggle/kaggle.json

# Verificar que la carpeta de experimentos exista
mkdir -p /content/buckets/b1/exp

echo "Configuración de entorno y credenciales Kaggle completada."

---

#### 2. Proceso de Subida y Marcado de Archivos

Esta parte se corre con el runtime en lenguaje **R**:
<br>Ir al menú: `Runtime` -> `Change runtime type` -> **R**

In [ ]:
# Limpieza del ambiente de R
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

format(Sys.time(), "%a %b %d %X %Y")

#### 3. Configuración de Parámetros y Directorios

In [ ]:
# Parámetros configurables
PARAM <- list()
PARAM$experimento <- "exp420_04"            # Nombre del experimento
PARAM$competencia <- "utn-2026-inicial"      # Nombre de la competencia en Kaggle
PARAM$prefijo_marcado <- "subido_"           # Prefijo que se le antepone al archivo tras ser subido (ej: 'subido_' o 'zz_')
PARAM$pausa_segundos <- 2                    # Pausa (seg) entre envíos para respetar límites de la API

# Determinar directorio base del experimento con soporte Colab y local
dir_base <- if (dir.exists("/content/buckets/b1/exp")) {
  "/content/buckets/b1/exp"
} else if (dir.exists("/workspace/exp")) {
  "/workspace/exp"
} else {
  file.path(getwd(), "exp")
}

dir_exp <- file.path(dir_base, PARAM$experimento)
dir_local <- file.path(dir_exp, "procesamiento_local")

cat("Directorio base:              ", dir_base, "\n")
cat("Directorio del experimento:   ", dir_exp, "\n")
cat("Directorio procesamiento_local:", dir_local, "\n")
cat("Competencia destino Kaggle:   ", PARAM$competencia, "\n")

#### 4. Búsqueda y Filtrado de Archivos Pendientes

In [ ]:
if (!dir.exists(dir_local)) {
  stop(sprintf("El directorio no existe: %s", dir_local))
}

# Obtener lista de todos los archivos CSV en procesamiento_local
todos_archivos <- list.files(dir_local, pattern = "\\.csv$", full.names = FALSE)

# Identificar los que ya fueron subidos/marcados (prefijo 'subido_' o 'zz_')
patron_ya_subidos <- paste0("^(", PARAM$prefijo_marcado, "|zz_)")
archivos_pendientes <- todos_archivos[!grepl(patron_ya_subidos, todos_archivos)]
archivos_ya_subidos <- todos_archivos[grepl(patron_ya_subidos, todos_archivos)]

cat("=========================================\n")
cat(sprintf("Total archivos CSV en directorio: %d\n", length(todos_archivos)))
cat(sprintf("Archivos ya subidos / marcados:   %d\n", length(archivos_ya_subidos)))
cat(sprintf("Archivos pendientes de subida:    %d\n", length(archivos_pendientes)))
cat("=========================================\n")

if (length(archivos_pendientes) > 0) {
  cat("\nPrimeros archivos pendientes a procesar:\n")
  print(head(archivos_pendientes, 10))
}

#### 5. Subida a Kaggle y Renombrado de Archivos

Para cada archivo pendiente:
1. Se ejecuta el comando `kaggle competitions submit`.
2. Se verifica la respuesta de Kaggle.
3. Si la subida fue exitosa, se renombra el archivo agregándole el prefijo de marcado (`subido_` o `zz_`).

In [ ]:
# Tabla para registrar el historial de subidas
tb_envios <- data.frame(
  iter = integer(),
  archivo_original = character(),
  archivo_nuevo = character(),
  estado = character(),
  mensaje_salida = character(),
  stringsAsFactors = FALSE
)

if (length(archivos_pendientes) == 0) {
  cat("No hay archivos pendientes para subir a Kaggle.\n")
} else {
  cat(sprintf("Iniciando subida de %d archivos a Kaggle...\n\n", length(archivos_pendientes)))

  for (i in seq_along(archivos_pendientes)) {
    arch <- archivos_pendientes[i]
    ruta_origen <- file.path(dir_local, arch)
    nuevo_nombre <- paste0(PARAM$prefijo_marcado, arch)
    ruta_destino <- file.path(dir_local, nuevo_nombre)

    cat(sprintf("[%d/%d] Subiendo: %s\n", i, length(archivos_pendientes), arch))

    # Armar comando Kaggle
    comando <- "kaggle competitions submit"
    competencia <- paste("-c", PARAM$competencia)
    arch_flag <- paste("-f", shQuote(ruta_origen))
    mensaje_flag <- paste0("-m '", arch, "'")
    linea <- paste(comando, competencia, arch_flag, mensaje_flag)

    # Ejecutar subida mediante system
    salida <- tryCatch(
      system(linea, intern = TRUE),
      error = function(e) paste("ERROR:", e$message)
    )

    texto_salida <- paste(salida, collapse = " | ")
    cat("  Respuesta Kaggle:", texto_salida, "\n")

    # Comprobar si hubo error
    es_error <- grepl("ERROR|400|404|429|Unauthorized|Could not find|Exception|Traceback", texto_salida, ignore.case = TRUE)

    if (!es_error) {
      # Renombrar archivo para marcarlo como subido
      exito_rename <- file.rename(ruta_origen, ruta_destino)
      if (exito_rename) {
        cat(sprintf("  -> [OK] Marcado como: %s\n\n", nuevo_nombre))
        estado <- "SUBIDO_Y_RENOMBRADO"
      } else {
        cat(sprintf("  -> [ALERTA] Subido a Kaggle pero no se pudo renombrar a: %s\n\n", nuevo_nombre))
        estado <- "SUBIDO_SIN_RENOMBRAR"
      }
    } else {
      cat(sprintf("  -> [ERROR] Falló el envío de %s\n\n", arch))
      estado <- "ERROR_KAGGLE"
    }

    tb_envios <- rbind(
      tb_envios,
      data.frame(
        iter = i,
        archivo_original = arch,
        archivo_nuevo = if (estado == "SUBIDO_Y_RENOMBRADO") nuevo_nombre else arch,
        estado = estado,
        mensaje_salida = texto_salida,
        stringsAsFactors = FALSE
      )
    )

    # Pausa entre peticiones
    if (i < length(archivos_pendientes) && PARAM$pausa_segundos > 0) {
      Sys.sleep(PARAM$pausa_segundos)
    }
  }

  cat("\n=========================================\n")
  cat("RESUMEN FINAL DE SUBIDAS:\n")
  cat(sprintf("Total procesados: %d\n", nrow(tb_envios)))
  cat(sprintf("Exitosos:         %d\n", sum(tb_envios$estado == "SUBIDO_Y_RENOMBRADO")))
  cat(sprintf("Con errores:      %d\n", sum(tb_envios$estado != "SUBIDO_Y_RENOMBRADO")))
  cat("=========================================\n")
}

#### 6. Tabla Resumen de Ejecución

In [ ]:
if (nrow(tb_envios) > 0) {
  tb_envios[, c("iter", "archivo_original", "archivo_nuevo", "estado")]
} else {
  cat("No se realizaron envíos en esta ejecución.\n")
}

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")